In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, BertForTokenClassification, AdamW, get_linear_schedule_with_warmup
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import json
import os

In [2]:
def build_label(train_data):
    uniquetypes = set()
    for item in train_data:
        for span in item['span_list']:
            uniquetypes.add(span['type'])
    sortedtypes = sorted(list(uniquetypes))
    label_to_id = {"O": 0}
    for t in sortedtypes:
        label_to_id[f"B-{t}"] = len(label_to_id)
        label_to_id[f"I-{t}"] = len(label_to_id)
    id_to_label = {values:keys for keys,values in label_to_id.items()}
    print(label_to_id)
    return label_to_id, id_to_label

In [3]:
file_path = 'train.jsonline'
def loaddata(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

In [ ]:
class Positiondataset(Dataset):
    def __init__(self, data, tokenizer, label_to_id, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.label_to_id = label_to_id
        self.max_len = max_len
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        line = self.data[idx]
        tokens = line['tokens']
        spans = line['span_list']
        raw_tags = ["O"] * len(tokens)
        for span in spans:
            start, end, t = span['start'], span['end'], span['type']
            raw_tags[start] = f"B-{t}"
            for i in range(start + 1, end + 1):
                raw_tags[i] = f"I-{t}"
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            return_offsets_mapping=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        labels = []
        word_ids = encoding.word_ids() 
        for idx in word_ids:
            if idx is None:
                labels.append(-100)
            else:
                labels.append(self.label_to_id[raw_tags[idx]])
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(labels)
        item.pop('offset_mapping')
        return item

In [ ]:
train_data = loaddata('train.jsonline')
dev_data = loaddata('train.jsonline')  
test_data = loaddata('test_public.jsonline')
label_to_id, id_to_label = build_label(train_data)
tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
train_dataset = Positiondataset(train_data, tokenizer, label_to_id)
dev_dataset = Positiondataset(dev_data, tokenizer, label_to_id)
test_dataset = Positiondataset(test_data, tokenizer, label_to_id)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
numlabels = len(label_to_id)

for i in range(3):
    print(f"数据实例{i+1}: {train_data[i]}")
model = BertForTokenClassification.from_pretrained(
    "bert-base-chinese",
    num_labels=numlabels,
    attention_probs_dropout_prob=0.09,
    hidden_dropout_prob=0.11
).to("cuda")
num_epochs = 6
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-5, weight_decay=0.01, eps=1e-8, betas=(0.98, 0.989))
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

{'O': 0, 'B-Ag': 1, 'I-Ag': 2, 'B-Dg': 3, 'I-Dg': 4, 'B-Ng': 5, 'I-Ng': 6, 'B-Tg': 7, 'I-Tg': 8, 'B-Vg': 9, 'I-Vg': 10, 'B-a': 11, 'I-a': 12, 'B-ad': 13, 'I-ad': 14, 'B-an': 15, 'I-an': 16, 'B-b': 17, 'I-b': 18, 'B-c': 19, 'I-c': 20, 'B-d': 21, 'I-d': 22, 'B-dq': 23, 'I-dq': 24, 'B-e': 25, 'I-e': 26, 'B-f': 27, 'I-f': 28, 'B-g': 29, 'I-g': 30, 'B-h': 31, 'I-h': 32, 'B-i': 33, 'I-i': 34, 'B-j': 35, 'I-j': 36, 'B-k': 37, 'I-k': 38, 'B-l': 39, 'I-l': 40, 'B-m': 41, 'I-m': 42, 'B-n': 43, 'I-n': 44, 'B-nr': 45, 'I-nr': 46, 'B-ns': 47, 'I-ns': 48, 'B-nt': 49, 'I-nt': 50, 'B-nz': 51, 'I-nz': 52, 'B-p': 53, 'I-p': 54, 'B-q': 55, 'I-q': 56, 'B-r': 57, 'I-r': 58, 'B-s': 59, 'I-s': 60, 'B-t': 61, 'I-t': 62, 'B-u': 63, 'I-u': 64, 'B-v': 65, 'I-v': 66, 'B-vd': 67, 'I-vd': 68, 'B-vn': 69, 'I-vn': 70, 'B-w': 71, 'I-w': 72, 'B-x': 73, 'I-x': 74, 'B-y': 75, 'I-y': 76, 'B-z': 77, 'I-z': 78}
数据实例1: {'tokens': ['二', '○', '○', '○', '年', '贺', '词'], 'span_list': [{'start': 0, 'end': 4, 'type': 't'}, {'start'

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
def evaluate(model, dataloader, id_to_label, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            for i in range(len(preds)):
                pred_labels = []
                true_labels = []
                for j in range(len(preds[i])):
                    if labels[i][j] != -100:
                        pred_labels.append(id_to_label[preds[i][j].item()])
                        true_labels.append(id_to_label[labels[i][j].item()])
                all_preds.append(pred_labels)
                all_labels.append(true_labels)
    report = classification_report(all_labels, all_preds, output_dict=True)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    return f1, precision, recall, report

In [12]:
device = torch.device("cuda")
model.to(device)
bestf1 = 0.0
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    model.train()
    total_loss = 0
    for i, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)
    print(f"Loss: {train_loss:.4f}")
    f1, precision, recall, report = evaluate(model, dev_loader, id_to_label, device)
    print(f"Dev F1: {f1:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")
    if f1 > bestf1:
        bestf1 = f1
        model.save_pretrained("model")
        tokenizer.save_pretrained("model")

Epoch 1/6
Loss: 0.8850


f:\conda\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Dev F1: 0.9408, Precision: 0.9360, Recall: 0.9455
Epoch 2/6
Loss: 0.1564
Dev F1: 0.9620, Precision: 0.9591, Recall: 0.9650
Epoch 3/6
Loss: 0.1087
Dev F1: 0.9718, Precision: 0.9707, Recall: 0.9730
Epoch 4/6
Loss: 0.0810
Dev F1: 0.9790, Precision: 0.9782, Recall: 0.9797
Epoch 5/6
Loss: 0.0628
Dev F1: 0.9834, Precision: 0.9829, Recall: 0.9840
Epoch 6/6
Loss: 0.0508
Dev F1: 0.9857, Precision: 0.9852, Recall: 0.9862


In [7]:
position2chinese = {
    'n': '名词', 't': '时间词', 's': '处所词', 'f': '方位词', 'm': '数词', 'q': '量词',
    'b': '区别词', 'r': '代词', 'v': '动词', 'a': '形容词', 'z': '状态词', 'd': '副词',
    'p': '介词', 'c': '连词', 'u': '助词', 'y': '语气词', 'e': '叹词', 'o': '拟声词',
    'i': '成语', 'l': '习用语', 'j': '简称', 'h': '前接成分', 'k': '后接成分',
    'g': '语素', 'x': '非语素字', 'w': '标点符号',
    'nr': '人名', 'ns': '地名', 'nt': '团体机关单位名称', 'nz': '其他专有名词',
    'Ng': '名语素', 'Vg': '动语素', 'Ag': '形容语素', 'Tg': '时语素', 'Dg': '副语素',
    'vn': '名动词', 'an': '名形词', 'vd': '副动词', 'ad': '副形词','dq':'副词性语素'
}
def get_pos_meaning(pos_tag):
    if pos_tag == 'O':
        return '未标注' 
    if pos_tag.startswith('B-'):
        return 'B-' + position2chinese.get(pos_tag[2:], f'未知标签:{pos_tag[2:]}')
    elif pos_tag.startswith('I-'):
        return 'I-' + position2chinese.get(pos_tag[2:], f'未知标签:{pos_tag[2:]}')
    else:
        return position2chinese.get(pos_tag, f'未知标签:{pos_tag}')

In [ ]:
def predict_pos(sentence, model, tokenizer, id_to_label, device, max_len=128):
        model.eval()
        tokens = tokenizer.tokenize(sentence)
        if len(tokens) > max_len - 2:
            tokens = tokens[:max_len - 2]

        tokens = ['[CLS]'] + tokens + ['[SEP]']
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_ids)
        padding_length = max_len - len(input_ids)
        
        input_ids += [tokenizer.pad_token_id] * padding_length
        attention_mask += [0] * padding_length
        
        input_ids = torch.tensor([input_ids]).to(device)
        attention_mask = torch.tensor([attention_mask]).to(device)
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)[0]
        predicted_labels = []
        for i, token_id in enumerate(preds):
            if i >= len(tokens):
                break
            if tokens[i] in ['[CLS]', '[SEP]']:
                continue
            label = id_to_label[token_id.item()]
            predicted_labels.append((tokens[i], label))
        return predicted_labels
def merge_bio_predictions(predictions):
    word_predictions = []
    curr_word = ''
    curr_tag = None
    for token, label in predictions:
        token = token[2:] if token.startswith('##') else token
        if label == 'O':
            if curr_word:
                word_predictions.append((curr_word, curr_tag))
                curr_word = ''
                curr_tag = None
            word_predictions.append((token, label))
            continue
        prefix, tag = label.split('-', 1) if '-' in label else ('B', label)
        if prefix == 'B' or not curr_word or curr_tag != tag:
            if curr_word:
                word_predictions.append((curr_word, curr_tag))
            curr_word = token
            curr_tag = tag
        else:
            curr_word += token
    if curr_word:
        word_predictions.append((curr_word, curr_tag))
    return word_predictions
model_path = "model"
device = torch.device("cuda")
model = BertForTokenClassification.from_pretrained(model_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)
testsentence = "为响应国家人工智能发展战略，中山大学于2020年正式成立人工智能学院。"
predictions = predict_pos(testsentence, model, tokenizer, id_to_label, device)
word_predictions = merge_bio_predictions(predictions)
print(f"句子: {testsentence}")
print(f"初始结果: {predictions}")
print("词语级标注结果:")
print(f"词语级标注结果: {word_predictions}")
for word, tag in word_predictions:
    meaning = get_pos_meaning(tag)
    print(f"{word}({meaning})", end=' ')

句子: 为响应国家人工智能发展战略，中山大学于2020年正式成立人工智能学院。
初始结果: [('为', 'B-p'), ('响', 'B-v'), ('应', 'I-v'), ('国', 'B-n'), ('家', 'I-n'), ('人', 'B-n'), ('工', 'I-n'), ('智', 'I-n'), ('能', 'I-n'), ('发', 'B-vn'), ('展', 'I-vn'), ('战', 'B-n'), ('略', 'I-n'), ('，', 'B-w'), ('中', 'B-nt'), ('山', 'I-nt'), ('大', 'I-nt'), ('学', 'I-nt'), ('于', 'B-p'), ('2020', 'B-t'), ('年', 'I-t'), ('正', 'B-ad'), ('式', 'I-ad'), ('成', 'B-v'), ('立', 'I-v'), ('人', 'B-n'), ('工', 'I-n'), ('智', 'I-n'), ('能', 'I-n'), ('学', 'B-n'), ('院', 'I-n'), ('。', 'B-w')]
词语级标注结果:
词语级标注结果: [('为', 'p'), ('响应', 'v'), ('国家', 'n'), ('人工智能', 'n'), ('发展', 'vn'), ('战略', 'n'), ('，', 'w'), ('中山大学', 'nt'), ('于', 'p'), ('2020年', 't'), ('正式', 'ad'), ('成立', 'v'), ('人工智能', 'n'), ('学院', 'n'), ('。', 'w')]
为(介词) 响应(动词) 国家(名词) 人工智能(名词) 发展(名动词) 战略(名词) ，(标点符号) 中山大学(团体机关单位名称) 于(介词) 2020年(时间词) 正式(副形词) 成立(动词) 人工智能(名词) 学院(名词) 。(标点符号) 
